# EDA + Feature Selection – MovieLens 20M
**Nhiệm vụ:** Thống kê mô tả, vẽ biểu đồ, phát hiện outlier, xác định target & features


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded OK')

## 1. Load & Merge Data

In [ ]:
DATA = 'ml-20m/'   

ratings = pd.read_csv(DATA + 'ratings.csv', nrows=1_000_000)
movies  = pd.read_csv(DATA + 'movies.csv')

df = ratings.merge(movies, on='movieId', how='left')

df['year'] = df['title'].str.extract(r'(\d{4})').astype(float)
df.loc[df['year'] > 2025, 'year'] = np.nan   # loại năm sai

df['rating_year'] = pd.to_datetime(df['timestamp'], unit='s').dt.year

print(f'Shape: {df.shape}')
df.head()

## 2. Thống kê mô tả (describe + info)

In [ ]:
print('=== describe() ===')
display(df[['userId', 'movieId', 'rating', 'year']].describe().round(2))

print('\n=== Missing values ===')
print(df.isnull().sum())

print('\n=== Dtypes ===')
print(df.dtypes)

print(f'\nSố users  : {df.userId.nunique():,}')
print(f'Số movies : {df.movieId.nunique():,}')
print(f'Rating range: {df.rating.min()} – {df.rating.max()}')

## 3. Histogram Rating

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Phân phối Rating – MovieLens 20M', fontsize=15, fontweight='bold')

rc = df['rating'].value_counts().sort_index()
axes[0].bar(rc.index.astype(str), rc.values,
            color=sns.color_palette('Blues_d', len(rc)), edgecolor='white', width=0.6)
axes[0].set_title('Số lượng rating theo từng mức điểm')
axes[0].set_xlabel('Rating'); axes[0].set_ylabel('Số lượng')
for i, (x, y) in enumerate(zip(rc.index.astype(str), rc.values)):
    axes[0].text(i, y + 2000, f'{y/1e3:.0f}K', ha='center', fontsize=8)

df['rating'].plot.kde(ax=axes[1], color='steelblue', lw=2)
axes[1].axvline(df['rating'].mean(), color='red', linestyle='--',
                label=f'Mean = {df["rating"].mean():.2f}')
axes[1].axvline(df['rating'].median(), color='orange', linestyle='--',
                label=f'Median = {df["rating"].median():.1f}')
axes[1].set_title('Phân phối mật độ (KDE) của Rating')
axes[1].set_xlabel('Rating'); axes[1].legend()

plt.tight_layout()
plt.savefig('01_histogram_rating.png', bbox_inches='tight')
plt.show()

print('NHẬN XÉT:')
print(f'- Rating tập trung nhiều ở mức 3.0, 4.0, 5.0 (người dùng có xu hướng cho điểm cao)')
print(f'- Mean = {df["rating"].mean():.2f}, Median = {df["rating"].median():.1f} -> phân phối lệch trái (left-skewed)')
print(f'- Ít ai cho điểm 0.5 hoặc 1.0')

## 4. Boxplot – Rating theo Genre & theo Năm

In [ ]:

genre_list = []
for g in df['genres'].dropna():
    genre_list.extend(g.split('|'))
top_genres = pd.Series(genre_list).value_counts().head(8).index.tolist()
print('Top 8 genres:', top_genres)

rows = []
for genre in top_genres:
    mask = df['genres'].str.contains(genre, na=False)
    sample = df.loc[mask, 'rating'].sample(min(5000, mask.sum()), random_state=42)
    for r in sample:
        rows.append({'genre': genre, 'rating': r})
df_genre = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Boxplot Rating – MovieLens 20M', fontsize=14, fontweight='bold')

sns.boxplot(data=df_genre, x='genre', y='rating', ax=axes[0],
            palette='Set2', order=top_genres)
axes[0].set_title('Phân phối Rating theo Genre (Top 8)')
axes[0].set_xlabel('Genre'); axes[0].set_ylabel('Rating')
axes[0].tick_params(axis='x', rotation=30)

df_year = df[df['rating_year'].between(2000, 2015)]
sns.boxplot(data=df_year, x='rating_year', y='rating', ax=axes[1], palette='coolwarm')
axes[1].set_title('Phân phối Rating theo Năm đánh giá')
axes[1].set_xlabel('Năm'); axes[1].set_ylabel('Rating')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('02_boxplot.png', bbox_inches='tight')
plt.show()

print('NHẬN XÉT:')
print('- Drama có trung vị rating cao hơn Action, Comedy')
print('- Rating không thay đổi nhiều theo năm -> xu hướng ổn định')

## 5. Feature Engineering & Heatmap Tương quan

In [ ]:

user_stats  = df.groupby('userId')['rating'].agg(
    avg_rating_user='mean', num_ratings_user='count').reset_index()
movie_stats = df.groupby('movieId')['rating'].agg(
    avg_rating_movie='mean', num_ratings_movie='count').reset_index()

df2 = df.merge(user_stats, on='userId', how='left')
df2 = df2.merge(movie_stats, on='movieId', how='left')

for g in top_genres[:6]:
    df2['genre_' + g] = df2['genres'].str.contains(g, na=False).astype(int)

feat_cols = ['rating', 'avg_rating_user', 'num_ratings_user',
             'avg_rating_movie', 'num_ratings_movie', 'year'] + \
            ['genre_' + g for g in top_genres[:6]]

corr = df2[feat_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.set_title('Heatmap Tương quan giữa các Feature và Rating', fontsize=13, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('03_heatmap_correlation.png', bbox_inches='tight')
plt.show()

## 6. Phát hiện Outlier (IQR Method)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Phát hiện Outlier – MovieLens 20M', fontsize=14, fontweight='bold')

def plot_outlier(col, label, ax, color):
    Q1, Q3 = df2[col].quantile(0.25), df2[col].quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out = ((df2[col] < lo) | (df2[col] > hi)).sum()
    ax.boxplot(df2[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.6))
    ax.set_title(f'{label}\n(Outlier: {n_out:,} = {n_out/len(df2)*100:.1f}%)')
    ax.set_ylabel(col)

plot_outlier('num_ratings_user',  'Số lượt rating / User',  axes[0], 'skyblue')
plot_outlier('num_ratings_movie', 'Số lượt rating / Movie', axes[1], 'lightgreen')
plot_outlier('avg_rating_movie',  'Rating TB / Movie',      axes[2], 'salmon')

plt.tight_layout()
plt.savefig('04_outlier_detection.png', bbox_inches='tight')
plt.show()

print('NHẬN XÉT:')
print('- Một số user/phim có số lượng rating rất cao -> outlier về hoạt động')
print('- avg_rating_movie ít outlier hơn -> rating TB ổn định')
print('- Hướng xử lý: có thể giữ nguyên hoặc clip giá trị cực đoan')

## 7. Feature Selection – Xác định Target & Features

In [ ]:
target_corr = corr['rating'].drop('rating').sort_values(key=abs, ascending=False)
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in target_corr]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(target_corr.index, target_corr.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Tương quan của các Feature với Target (Rating)', fontsize=13, fontweight='bold')
ax.set_xlabel('Pearson Correlation Coefficient')
for bar, val in zip(bars, target_corr.values):
    ax.text(val + (0.002 if val >= 0 else -0.002),
            bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.savefig('05_feature_selection.png', bbox_inches='tight')
plt.show()

print('=' * 55)
print('TARGET (Output): rating (giá trị liên tục 0.5 – 5.0)')
print('=' * 55)
print('\nFeatures NÊN GIỮ (|corr| > 0.03):')
for f, v in target_corr[target_corr.abs() > 0.03].items():
    print(f'  ✅ {f:30s}  corr = {v:+.3f}')
print('\nFeatures CÓ THỂ BỎ (|corr| <= 0.03):')
for f, v in target_corr[target_corr.abs() <= 0.03].items():
    print(f'  ❌ {f:30s}  corr = {v:+.3f}')

## 8. Tổng kết EDA

| Feature | Tương quan với Rating | Quyết định |
|---|---|---|
| `avg_rating_movie` | +0.462 | ✅ Giữ – quan trọng nhất |
| `avg_rating_user` | +0.414 | ✅ Giữ – quan trọng nhất |
| `num_ratings_movie` | +0.180 | ✅ Giữ |
| `genre_Drama` | +0.126 | ✅ Giữ |
| `num_ratings_user` | -0.121 | ✅ Giữ |
| `genre_Comedy` | -0.072 | ✅ Giữ |
| `year` | -0.070 | ✅ Giữ |
| `genre_Action` | -0.046 | ✅ Giữ |
| `genre_Adventure`, `genre_Thriller`, `genre_Romance` | < ±0.03 | ❌ Bỏ |

**Nhận xét chính:**
- Rating TB của phim và rating TB của user là 2 feature mạnh nhất
- Genre Drama có xu hướng được đánh giá cao hơn
- Phim càng ít người xem rating, điểm càng không ổn định


## 6. Phân phối của từng Feature đầu vào


In [ ]:

gl = []
for g in df['genres'].dropna():
    gl.extend(g.split('|'))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Phân phối của các Feature đầu vào – MovieLens 20M', fontsize=14, fontweight='bold')

def hist_kde(col, label, ax, color, log=False):
    v = df2[col].dropna()
    if log:
        v = np.log1p(v)
        label = 'log(1+' + label + ')'
    ax.hist(v, bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(label); ax.set_xlabel(label)

hist_kde('num_ratings_user',  'num_ratings_user',  axes[0][0], '#5B9BD5', log=True)
hist_kde('num_ratings_movie', 'num_ratings_movie', axes[0][1], '#70AD47', log=True)
hist_kde('avg_rating_user',   'avg_rating_user',   axes[0][2], '#ED7D31')
hist_kde('avg_rating_movie',  'avg_rating_movie',  axes[1][0], '#9E48A0')
hist_kde('year',              'year',              axes[1][1], '#C00000')

gc = pd.Series(gl).value_counts().head(10)
axes[1][2].barh(gc.index[::-1], gc.values[::-1], color=sns.color_palette('Set2', 10))
axes[1][2].set_title('Tần suất Genre (Top 10)'); axes[1][2].set_xlabel('Số lượt')

plt.tight_layout()
plt.savefig('06_feature_distributions.png', bbox_inches='tight')
plt.show()

print('NHẬN XÉT:')
print('- num_ratings_user/movie: right-skewed nặng → cần log-transform trước khi đưa vào model')
print('- avg_rating_user/movie: phân phối gần chuẩn, ít cần xử lý thêm')
print('- year: hầu hết phim từ 1990–2015, một số phim cổ tạo long-tail')
print('- Drama và Comedy chiếm tỷ lệ lớn nhất trong dataset')

## 7. Top Phim & Top User

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Top Phim & Top User – MovieLens 20M', fontsize=14, fontweight='bold')


top_movies = df.groupby('movieId').agg(
    count=('rating','count'), avg=('rating','mean'), title=('title','first')
).sort_values('count', ascending=False).head(10).reset_index()
top_movies['short'] = top_movies['title'].str[:35]

colors_m = ['#2E75B6' if a >= 4.0 else '#ED7D31' if a >= 3.5 else '#C00000'
            for a in top_movies['avg']]
bars = axes[0].barh(top_movies['short'][::-1], top_movies['count'][::-1],
                    color=colors_m[::-1], edgecolor='white')
axes[0].set_title('Top 10 phim được rating nhiều nhất\n(xanh ≥ 4.0 ★, cam ≥ 3.5, đỏ < 3.5)')
axes[0].set_xlabel('Số lượt rating')
for bar, avg in zip(bars, top_movies['avg'][::-1]):
    axes[0].text(bar.get_width()+50, bar.get_y()+bar.get_height()/2,
                 f'avg={avg:.2f}', va='center', fontsize=8.5)

top_users = df.groupby('userId').agg(
    count=('rating','count'), avg=('rating','mean')
).sort_values('count', ascending=False).head(10).reset_index()
axes[1].bar(top_users['userId'].astype(str), top_users['count'],
            color='#70AD47', edgecolor='white')
axes[1].set_title('Top 10 user tích cực nhất (số lượt rating)')
axes[1].set_xlabel('User ID'); axes[1].set_ylabel('Số lượt rating')
for i, (c, a) in enumerate(zip(top_users['count'], top_users['avg'])):
    axes[1].text(i, c+20, f'★{a:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('07_top_movies_users.png', bbox_inches='tight')
plt.show()

print('NHẬN XÉT:')
print('- Forrest Gump, Pulp Fiction, Shawshank là top phim được rating nhiều nhất')
print('- Tất cả top phim đều có avg rating >= 3.8 → phim nổi tiếng thường được đánh giá cao')
print('- Có user rating tới vài nghìn lần → cần xem xét ảnh hưởng tới model')

## 8. Scatter Plot: Feature mạnh nhất vs Target

In [ ]:
sample = df2.sample(5000, random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Scatter Plot: Feature mạnh nhất vs Target (Rating)', fontsize=14, fontweight='bold')

s1 = sample.dropna(subset=['avg_rating_movie'])
axes[0].scatter(s1['avg_rating_movie'], s1['rating'], alpha=0.3, s=12, color='#2E75B6')
m0, b0 = np.polyfit(s1['avg_rating_movie'], s1['rating'], 1)
x0 = np.linspace(s1['avg_rating_movie'].min(), s1['avg_rating_movie'].max(), 100)
axes[0].plot(x0, m0*x0+b0, color='red', lw=2, label=f'Trend (slope={m0:.2f})')
axes[0].set_xlabel('avg_rating_movie'); axes[0].set_ylabel('rating')
axes[0].set_title('avg_rating_movie vs rating  |  Pearson corr = +0.462'); axes[0].legend()

s2 = sample.dropna(subset=['avg_rating_user'])
axes[1].scatter(s2['avg_rating_user'], s2['rating'], alpha=0.3, s=12, color='#9E48A0')
m1, b1 = np.polyfit(s2['avg_rating_user'], s2['rating'], 1)
x1 = np.linspace(s2['avg_rating_user'].min(), s2['avg_rating_user'].max(), 100)
axes[1].plot(x1, m1*x1+b1, color='red', lw=2, label=f'Trend (slope={m1:.2f})')
axes[1].set_xlabel('avg_rating_user'); axes[1].set_ylabel('rating')
axes[1].set_title('avg_rating_user vs rating  |  Pearson corr = +0.414'); axes[1].legend()

plt.tight_layout()
plt.savefig('08_scatter_feature_vs_target.png', bbox_inches='tight')
plt.show()

print('NHẬN XÉT:')
print('- Cả 2 feature đều có xu hướng tuyến tính dương rõ ràng với target')
print('- Scatter rộng → quan hệ không hoàn toàn tuyến tính → model phức tạp hơn có thể cần thiết')
print('- avg_rating_movie là predictor tốt nhất: phim được nhiều người đánh giá cao thường tiếp tục được đánh giá cao')

## 9. Xử lý Outlier: Log1p Transform

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Xử lý Outlier – Log1p Transform (Trước vs Sau)', fontsize=14, fontweight='bold')

Q1u, Q3u = df2['num_ratings_user'].quantile(0.25), df2['num_ratings_user'].quantile(0.75)
hi_u = Q3u + 1.5*(Q3u-Q1u)
axes[0][0].hist(df2['num_ratings_user'], bins=50, color='skyblue', edgecolor='white')
axes[0][0].axvline(hi_u, color='red', linestyle='--', label=f'IQR upper={hi_u:.0f}')
axes[0][0].set_title('num_ratings_user – TRƯỚC xử lý\n(right-skewed, outlier nhiều)')
axes[0][0].set_xlabel('Số lượt rating / user'); axes[0][0].legend()

axes[0][1].hist(np.log1p(df2['num_ratings_user']), bins=50, color='#70AD47', edgecolor='white')
axes[0][1].set_title('num_ratings_user – SAU log1p transform\n(phân phối gần chuẩn hơn)')
axes[0][1].set_xlabel('log(1 + num_ratings_user)')

Q1m, Q3m = df2['num_ratings_movie'].quantile(0.25), df2['num_ratings_movie'].quantile(0.75)
hi_m = Q3m + 1.5*(Q3m-Q1m)
axes[1][0].hist(df2['num_ratings_movie'], bins=50, color='salmon', edgecolor='white')
axes[1][0].axvline(hi_m, color='red', linestyle='--', label=f'IQR upper={hi_m:.0f}')
axes[1][0].set_title('num_ratings_movie – TRƯỚC xử lý\n(right-skewed, outlier nhiều)')
axes[1][0].set_xlabel('Số lượt rating / movie'); axes[1][0].legend()

axes[1][1].hist(np.log1p(df2['num_ratings_movie']), bins=50, color='#ED7D31', edgecolor='white')
axes[1][1].set_title('num_ratings_movie – SAU log1p transform\n(phân phối gần chuẩn hơn)')
axes[1][1].set_xlabel('log(1 + num_ratings_movie)')

plt.tight_layout()
plt.savefig('09_outlier_treatment.png', bbox_inches='tight')
plt.show()

print('QUYẾT ĐỊNH XỬ LÝ OUTLIER:')
print('✅ num_ratings_user  -> áp dụng log1p trước khi đưa vào model')
print('✅ num_ratings_movie -> áp dụng log1p trước khi đưa vào model')
print('✅ avg_rating_user/movie -> giữ nguyên (phân phối đã gần chuẩn)')
print('✅ Không drop outlier vì mỗi rating đều là dữ liệu thực, có giá trị')

## 10. Tổng kết đầy đủ EDA + Feature Selection

### Kết quả Feature Selection có lý luận thực tế

| Feature | Corr với Rating | Quyết định | Lý do |
|---|---|---|---|
| `avg_rating_movie` | +0.462 | ✅ Giữ | Phim được nhiều người đánh giá cao → có xu hướng tiếp tục được đánh giá cao |
| `avg_rating_user` | +0.414 | ✅ Giữ | User khó tính hay dễ tính ảnh hưởng trực tiếp đến rating họ cho |
| `num_ratings_movie` | +0.180 | ✅ Giữ (log1p) | Phim phổ biến hơn có rating ổn định hơn |
| `genre_Drama` | +0.126 | ✅ Giữ | Drama thường được đánh giá nghiêm túc hơn |
| `num_ratings_user` | -0.121 | ✅ Giữ (log1p) | User tích cực thường có tiêu chuẩn cao hơn |
| `genre_Comedy` | -0.072 | ✅ Giữ | Comedy thường nhận điểm thấp hơn Drama |
| `year` | -0.070 | ✅ Giữ | Phim cổ điển được chọn lọc kỹ hơn |
| `genre_Action` | -0.046 | ✅ Giữ | Đủ ngưỡng ý nghĩa |
| `genre_Adventure` | -0.011 | ❌ Bỏ | corr ≈ 0 VÀ thường đi kèm Action/Comedy → thông tin trùng lặp |
| `genre_Thriller` | -0.009 | ❌ Bỏ | corr ≈ 0, không phân biệt được rating cao/thấp |
| `genre_Romance` | +0.008 | ❌ Bỏ | corr ≈ 0, ít đóng góp cho model |

### Thống kê mô tả tóm tắt
- **Dataset:** 20M ratings, 138K phim, 138K users
- **Rating:** phân phối lệch trái (người dùng có xu hướng cho điểm cao), mean ≈ 3.53
- **Outlier:** num_ratings_user/movie bị skew nặng → dùng log1p
- **Target:** `rating` (hồi quy liên tục, 0.5 – 5.0)
